# AI Harness Engineering 实训课 —— 演示代码

> **天大软件工程学院 · 软件工程专业大三**  
> **课程**：AI Harness Engineering  
> **本 Notebook 目标**：展示如何定义自定义工具（Tool）并让 LLM 调用它们

---

## 目录

1. [环境准备](#1-环境准备)
2. [理解工具调用的核心概念](#2-理解工具调用的核心概念)
3. [手动定义工具 Schema（OpenAI 格式）](#3-手动定义工具-schemaopenai-格式)
4. [模拟工具调度器](#4-模拟工具调度器)
5. [完整的 Agent Loop 演示](#5-完整的-agent-loop-演示)
6. [使用 LangChain 定义工具](#6-使用-langchain-定义工具)
6.1 [认识 OpenClaw —— SKILL.md 技能生态](#61-认识-openclaw--skillmd-技能生态)
7. [思考与练习](#7-思考与练习)

---
## 1. 环境准备

本 Notebook 需要 `openai` 和 `langchain-core` 两个包。  
如果你还没有安装，请取消下面单元格的注释并运行。

In [13]:
# 如果尚未安装依赖，请取消下一行注释后运行。
# %pip install openai langchain-core -q

In [14]:
import json
import os
import sys
from typing import Any, Callable, Dict, List, Optional

print("环境准备完成！")
print(f"Python 版本: {sys.version}")

环境准备完成！
Python 版本: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]


---
## 2. 理解工具调用的核心概念

### 什么是工具调用（Tool Calling / Function Calling）？

工具调用是让 LLM **自主决定**何时、如何调用外部函数的机制。流程如下：

```
用户提问 → LLM 推理 → 判断需要调用工具
                         ↓
                  输出 tool_call: {name: "web_search", args: {query: "天津天气"}}
                         ↓
                  工具调度器执行 web_search("天津天气")
                         ↓
                  返回结果 "天津今日晴，25°C"
                         ↓
                  LLM 结合工具结果生成最终回答
```

### 工具定义的三个核心要素

1. **Schema（模式）**：告诉 LLM 这个工具叫什么、做什么、需要什么参数
2. **Handler（处理器）**：实际执行逻辑的函数
3. **Registry（注册表）**：把 Schema 和 Handler 绑定在一起

---
## 3. 手动定义工具 Schema（OpenAI 格式）

OpenAI 的 Function Calling 使用 JSON Schema 格式描述工具。  
这是所有 Agent 框架的基础格式——OpenClaw、Hermes Agent、LangChain 都兼容这种格式。

In [15]:
# ==========================================
# 工具 1：web_search —— 搜索互联网获取信息
# ==========================================

web_search_schema = {
    "type": "function",
    "function": {
        # 工具名称：LLM 通过这个名称来调用工具
        # 命名建议：动词_名词，如 web_search, read_file, calculate
        "name": "web_search",
        
        # 工具描述：这是 LLM 判断"是否需要调用此工具"的唯一依据！
        # 描述必须清晰、准确、无歧义
        "description": "搜索互联网获取实时信息。当用户询问时事、天气、股价等需要最新数据的问题时使用。",
        
        # 参数定义：遵循 JSON Schema 格式
        "parameters": {
            "type": "object",
            "properties": {
                # 每个参数都需要 type 和 description
                "query": {
                    "type": "string",
                    "description": "搜索关键词，应该是一个简洁明确的查询词组"
                }
            },
            # required 数组指定哪些参数是必须的
            "required": ["query"]
        }
    }
}

print("✅ web_search 工具 Schema 定义完成")
print(json.dumps(web_search_schema, indent=2, ensure_ascii=False))

✅ web_search 工具 Schema 定义完成
{
  "type": "function",
  "function": {
    "name": "web_search",
    "description": "搜索互联网获取实时信息。当用户询问时事、天气、股价等需要最新数据的问题时使用。",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "搜索关键词，应该是一个简洁明确的查询词组"
        }
      },
      "required": [
        "query"
      ]
    }
  }
}


In [16]:
# ==========================================
# 工具 2：calculate —— 计算数学表达式
# ==========================================

calculate_schema = {
    "type": "function",
    "function": {
        "name": "calculate",
        # 注意描述中的"精确计算"：帮助 LLM 区分何时该用计算器 vs 自己估算
        "description": "精确计算数学表达式的结果。当需要进行数值计算、单位换算或复杂数学运算时使用。不支持文字搜索。",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    # 给出具体示例，减少 LLM 格式错误
                    "description": "数学表达式，如 '2+3*4'、'sqrt(16)'、'3.14*10**2'"
                }
            },
            "required": ["expression"]
        }
    }
}

print("✅ calculate 工具 Schema 定义完成")
print(json.dumps(calculate_schema, indent=2, ensure_ascii=False))

✅ calculate 工具 Schema 定义完成
{
  "type": "function",
  "function": {
    "name": "calculate",
    "description": "精确计算数学表达式的结果。当需要进行数值计算、单位换算或复杂数学运算时使用。不支持文字搜索。",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "数学表达式，如 '2+3*4'、'sqrt(16)'、'3.14*10**2'"
        }
      },
      "required": [
        "expression"
      ]
    }
  }
}


In [17]:
# ==========================================
# 工具 3：get_current_time —— 获取当前时间
# ==========================================

get_current_time_schema = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        # 描述中强调"不需要参数"，避免 LLM 传入多余参数
        "description": "获取当前的日期和时间。不需要任何参数。当用户询问现在几点、今天几号时使用。",
        "parameters": {
            "type": "object",
            # 没有参数时，properties 为空字典，required 为空列表
            "properties": {},
            "required": []
        }
    }
}

print("✅ get_current_time 工具 Schema 定义完成")
print(json.dumps(get_current_time_schema, indent=2, ensure_ascii=False))

✅ get_current_time 工具 Schema 定义完成
{
  "type": "function",
  "function": {
    "name": "get_current_time",
    "description": "获取当前的日期和时间。不需要任何参数。当用户询问现在几点、今天几号时使用。",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": []
    }
  }
}


### 💡 关键设计要点

| 要点 | 说明 | 反面教材 |
|------|------|---------|
| 描述要具体 | "搜索互联网获取实时信息" | "搜索东西" |
| 给出触发场景 | "当用户询问时事、天气时使用" | 无触发条件 |
| 参数要有示例 | "如 '2+3*4'、'sqrt(16)'" | 无示例 |
| 区分相似工具 | calculate: "精确计算" vs web_search: "搜索信息" | 两者描述模糊重叠 |

---
## 4. 模拟工具调度器

工具调度器（Tool Dispatcher）是 Harness 的核心组件之一。  
它负责：接收 LLM 的工具调用请求 → 找到对应的 Handler → 执行 → 返回结果。

In [18]:
# ==========================================
# 第二步：构建工具注册表（Tool Registry）
# ==========================================

class ToolRegistry:
    """
    工具注册表 —— Agent Harness 的核心组件之一。
    
    职责：
    1. 注册工具（绑定 Schema + Handler）
    2. 查找工具（根据名称找到对应 Handler）
    3. 获取所有工具的 Schema（供 LLM 使用）
    4. 执行工具调用（根据 LLM 的 tool_call 执行对应 Handler）
    
    这类似于 Hermes Agent 中的 tools/registry.py 和
    LangChain 中的 tool 绑定机制。
    """
    
    def __init__(self):
        # _tools: 名称 → {schema, handler} 的映射
        self._tools: Dict[str, Dict[str, Any]] = {}
    
    def register(self, schema: dict, handler: Callable) -> None:
        """
        注册一个工具。
        
        参数:
            schema: OpenAI 格式的工具 Schema
            handler: 实际执行函数，接收 dict 参数，返回 str 结果
        
        类比：
            - Hermes Agent: registry.register(name=..., schema=..., handler=...)
            - LangChain: llm.bind_tools([tool]) 自动注册
        """
        name = schema["function"]["name"]
        if name in self._tools:
            print(f"⚠️ 工具 '{name}' 已存在，将被覆盖")
        self._tools[name] = {"schema": schema, "handler": handler}
        print(f"✅ 工具 '{name}' 注册成功")
    
    def get_schemas(self) -> List[dict]:
        """
        获取所有已注册工具的 Schema 列表。
        这个列表会传递给 LLM，让它知道有哪些工具可用。
        """
        return [tool["schema"] for tool in self._tools.values()]
    
    def dispatch(self, tool_name: str, arguments: dict) -> str:
        """
        调度并执行工具调用。
        这是 Agent Loop 中的关键步骤：LLM 输出 tool_call → 调度器执行 → 返回结果。
        
        参数:
            tool_name: 工具名称（LLM 输出的 function name）
            arguments: 工具参数（LLM 输出的 function arguments）
        
        返回:
            工具执行结果的字符串
        """
        if tool_name not in self._tools:
            return json.dumps({"error": f"工具 '{tool_name}' 不存在"}, ensure_ascii=False)
        
        handler = self._tools[tool_name]["handler"]
        try:
            result = handler(**arguments)
            return result
        except Exception as e:
            return json.dumps({"error": f"工具执行失败: {str(e)}"}, ensure_ascii=False)
    
    def list_tools(self) -> None:
        """列出所有已注册的工具。"""
        print("\n📋 已注册工具列表：")
        print("-" * 60)
        for name, tool in self._tools.items():
            desc = tool["schema"]["function"]["description"]
            params = list(tool["schema"]["function"]["parameters"].get("properties", {}).keys())
            print(f"  {name}({', '.join(params)})")
            print(f"    → {desc[:50]}...")
        print("-" * 60)


# ==========================================
# 第三步：注册我们的三个工具
# ==========================================

registry = ToolRegistry()

# 注册工具：将 Schema 和 Handler 绑定
registry.register(web_search_schema, web_search_handler)
registry.register(calculate_schema, calculate_handler)
registry.register(get_current_time_schema, get_current_time_handler)

# 列出所有已注册的工具
registry.list_tools()

NameError: name 'web_search_handler' is not defined

In [ ]:
# ==========================================
# 第四步：测试工具调度
# ==========================================

print("--- 工具调度测试 ---\n")

# 模拟 LLM 输出的 tool_call
# 在真实场景中，这些是 LLM 根据用户问题和工具 Schema 自动生成的
mock_tool_calls = [
    {"name": "web_search", "arguments": {"query": "天津天气"}},
    {"name": "calculate", "arguments": {"expression": "2 + 3 * 4"}},
    {"name": "get_current_time", "arguments": {}},
]

for call in mock_tool_calls:
    print(f"🔔 LLM 请求调用工具: {call['name']}({call['arguments']})")
    result = registry.dispatch(call["name"], call["arguments"])
    print(f"📤 工具返回结果: {result}")
    print()

---
## 5. 完整的 Agent Loop 演示

现在我们把所有组件组合在一起，实现一个完整的 Agent Loop。  
由于我们在课堂上可能没有 API Key，这里用一个**模拟 LLM** 来演示流程。

In [ ]:
class SimpleAgent:
    """
    简易 Agent —— 展示 Agent Loop 的核心逻辑。

    在真实场景中，LLM 推理部分由 OpenAI API 等服务完成。
    这里使用规则匹配模拟 LLM 的工具选择逻辑，方便课堂无 API Key 时运行。
    """

    def __init__(self, registry: ToolRegistry, system_prompt: str = "你是一个有用的 AI 助手。"):
        self.registry = registry
        self.system_prompt = system_prompt
        self.messages: List[dict] = [{"role": "system", "content": system_prompt}]
        self.max_iterations = 5

    def _extract_search_query(self, user_message: str) -> str:
        query = user_message
        for phrase in ["帮我查一下", "查一下", "帮我搜", "搜索", "搜"]:
            query = query.replace(phrase, "")
        query = query.strip(" ，。？！")
        if "天气" in query and not any(city in query for city in ["天津", "北京", "上海", "广州", "深圳"]):
            query = "天津天气"
        return query or user_message

    def _extract_math_expression(self, user_message: str) -> str:
        import re
        cleaned = user_message.replace("计算", "").replace("等于多少", "").replace("等于", "")
        matches = re.findall(r"[0-9piPIeE+\-*/().^ ]+|sqrt\([^)]*\)|sin\([^)]*\)|cos\([^)]*\)", cleaned)
        expression = "".join(matches).strip()
        return expression or "1+1"

    def _mock_llm_inference(self, user_message: str) -> dict:
        """模拟一次 LLM 推理：根据用户输入选择工具或直接回答。"""
        user_lower = user_message.lower()

        if any(kw in user_lower for kw in ["天气", "搜索", "查", "搜"]):
            return {
                "type": "tool_call",
                "tool_name": "web_search",
                "arguments": {"query": self._extract_search_query(user_message)},
            }
        if any(kw in user_lower for kw in ["计算", "等于", "+", "-", "*", "/", "算"]):
            return {
                "type": "tool_call",
                "tool_name": "calculate",
                "arguments": {"expression": self._extract_math_expression(user_message)},
            }
        if any(kw in user_lower for kw in ["时间", "几点", "日期", "今天"]):
            return {
                "type": "tool_call",
                "tool_name": "get_current_time",
                "arguments": {},
            }
        return {
            "type": "text",
            "content": f"我理解你的问题是：{user_message}。这是一个不需要工具的直接回答。"
        }

    def _generate_final_response(self, user_message: str, tool_results: List[str]) -> str:
        """模拟 LLM 根据工具结果生成最终回答。"""
        if not tool_results:
            return f"关于你的问题「{user_message}」，我暂时没有找到相关信息。"

        responses = []
        for result_json in tool_results:
            try:
                result = json.loads(result_json)
                if result.get("success"):
                    if "expression" in result and "result" in result:
                        responses.append(f"{result['expression']} = {result['result']}")
                    elif "datetime" in result:
                        responses.append(f"现在是 {result['datetime']}，{result['weekday']}")
                    elif "result" in result:
                        responses.append(result["result"])
                else:
                    responses.append(f"工具执行失败：{result.get('error', '未知错误')}")
            except json.JSONDecodeError:
                responses.append(result_json)

        return f"根据工具结果：{'；'.join(str(r) for r in responses)}"

    def run(self, user_message: str) -> str:
        """
        Agent Loop 的主入口。

        教学版流程：
        1. 用户输入加入消息列表
        2. 模拟 LLM 判断是否需要工具
        3. 如需工具，执行工具并把结果写回消息列表
        4. 模拟 LLM 基于工具结果生成最终回答
        """
        print(f"\n{'='*60}")
        print(f"👤 用户: {user_message}")
        print(f"{'='*60}")

        self.messages.append({"role": "user", "content": user_message})

        tool_results = []
        llm_output = None

        for i in range(self.max_iterations):
            print(f"\n--- 迭代 {i+1} ---")

            if tool_results:
                print("🤖 LLM 读取工具结果，生成最终回答")
                break

            llm_output = self._mock_llm_inference(user_message)

            if llm_output["type"] == "tool_call":
                tool_name = llm_output["tool_name"]
                arguments = llm_output["arguments"]
                print(f"🤖 LLM 决定调用工具: {tool_name}({arguments})")

                result = self.registry.dispatch(tool_name, arguments)
                tool_results.append(result)
                print(f"🔧 工具返回: {result}")

                # 简化表示。真实 OpenAI tool message 还需要 tool_call_id。
                self.messages.append({
                    "role": "tool",
                    "name": tool_name,
                    "content": result
                })
                continue

            print(f"🤖 LLM 直接回答: {llm_output['content']}")
            self.messages.append({"role": "assistant", "content": llm_output["content"]})
            return llm_output["content"]

        final_response = self._generate_final_response(user_message, tool_results)
        self.messages.append({"role": "assistant", "content": final_response})
        print(f"\n🤖 最终回答: {final_response}")

        return final_response


# ==========================================
# 创建并运行 Agent
# ==========================================

agent = SimpleAgent(
    registry=registry,
    system_prompt="你是一个智能助手，可以搜索信息、计算数学表达式、查询时间。请用中文回答。"
)

print("\n" + "🔥" * 20)
print("Agent Loop 完整演示")
print("🔥" * 20)

agent.run("帮我查一下天津天气")
agent.run("计算 2 + 3 * 4")
agent.run("现在几点了？")
agent.run("你好，请介绍一下你自己")

In [ ]:
# ==========================================
# 查看完整的对话历史（短期记忆）
# ==========================================

print("\n" + "📝" * 20)
print("完整对话历史（Agent 的短期记忆）")
print("📝" * 20)

for i, msg in enumerate(agent.messages):
    role = msg["role"]
    content = msg.get("content", "")[:100]  # 截断显示
    name = msg.get("name", "")
    prefix = f"[{name}]" if name else ""
    print(f"{i+1}. [{role}]{prefix} {content}...")

print(f"\n总消息数: {len(agent.messages)}")
print("\n💡 思考：如果对话持续上百轮，这些消息会超出 LLM 的上下文窗口限制，该怎么办？")
print("   → 答案：上下文压缩（Context Compression）—— Hermes Agent 会在接近 token 上限时自动压缩历史消息")

---
## 6. 使用 LangChain 定义工具

上面的代码展示了工具调用的底层原理。  
实际开发中，我们通常使用框架提供的工具定义方式。这里演示 LangChain 的 `@tool` 装饰器。

In [ ]:
# ==========================================
# LangChain 工具定义方式
# ==========================================

try:
    from langchain_core.tools import tool as lc_tool
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False
    print("⚠️ langchain-core 未安装，将展示代码示例但不执行")


if LANGCHAIN_AVAILABLE:
    # ---- 方式 1：使用 @tool 装饰器（最常用）----
    @lc_tool
    def web_search(query: str) -> str:
        """搜索互联网获取实时信息。当用户询问时事、天气、股价等需要最新数据的问题时使用。
        
        Args:
            query: 搜索关键词，应该是一个简洁明确的查询词组
        """
        # 实际项目中替换为真实搜索 API 调用
        return f"搜索 '{query}' 的结果：[模拟] 找到了相关信息"

    @lc_tool
    def calculate(expression: str) -> str:
        """精确计算数学表达式的结果。当需要进行数值计算时使用。
        
        Args:
            expression: 数学表达式，如 '2+3*4'
        """
        try:
            result = _safe_eval_math(expression)
            return f"{expression} = {result}"
        except Exception as e:
            return f"计算错误: {e}"

    def dump_args_schema(lc_tool_obj):
        schema = lc_tool_obj.args_schema
        if hasattr(schema, "model_json_schema"):
            return schema.model_json_schema()
        return schema.schema()

    # 查看 LangChain 自动生成的工具 Schema
    print("✅ LangChain @tool 装饰器定义完成")
    print(f"\n--- web_search 工具 Schema ---")
    print(json.dumps(dump_args_schema(web_search), indent=2, ensure_ascii=False))
    print(f"\n--- calculate 工具 Schema ---")
    print(json.dumps(dump_args_schema(calculate), indent=2, ensure_ascii=False))
    
else:
    # 展示代码示例（即使没有安装也能看到）
    example_code = '''
# === LangChain @tool 装饰器示例 ===

from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    \"\"\"搜索互联网获取实时信息。

    Args:
        query: 搜索关键词
    \"\"\"
    return search_api(query)

@tool
def calculate(expression: str) -> str:
    \"\"\"精确计算数学表达式。

    Args:
        expression: 数学表达式
    \"\"\"
    return calculate_handler(expression)

# 绑定工具到 LLM
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools([web_search, calculate])

# 调用
result = llm_with_tools.invoke("天津今天天气怎么样？")
# result.tool_calls 包含 LLM 决定调用的工具列表
'''
    print(example_code)

In [ ]:
# ==========================================
# 对比：手写 Schema vs LangChain @tool vs OpenClaw SKILL.md
# ==========================================

print("""
╔══════════════════════════════════════════════════════════════════════════════════╗
║            手写 Schema vs LangChain @tool vs OpenClaw SKILL.md 对比            ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  手写 Schema（第3节）：                                                          ║
║  ✅ 完全控制格式和描述                                                          ║
║  ✅ 不依赖任何框架                                                             ║
║  ❌ 代码冗长，容易出错                                                         ║
║  ❌ 需要手动维护 Schema 与 Handler 的同步                                       ║
║                                                                                  ║
║  LangChain @tool（第6节）：                                                     ║
║  ✅ 从函数签名自动生成 Schema                                                   ║
║  ✅ 代码简洁，类型安全                                                         ║
║  ❌ 依赖框架                                                                   ║
║  ❌ 描述格式受限于 docstring                                                   ║
║                                                                                  ║
║  OpenClaw SKILL.md（第6.1节）：                                                  ║
║  ✅ 纯 Markdown，人人都能编写                                                   ║
║  ✅ 包含工作流程、触发条件、步骤清单，更接近"最佳实践"文档                   ║
║  ❌ 无自动 Schema 生成（依赖人工编写）                                         ║
║  ❌ 不直接映射为可执行代码（需配合 Shell/API Handler）                          ║
║                                                                                  ║
║  选择建议：                                                                     ║
║  - 学习阶段：手写 Schema → 理解底层原理                                        ║
║  - 开发阶段：使用框架 @tool → 提高开发效率                                      ║
║  - 知识复用：OpenClaw SKILL.md → 社区技能生态，5,400+ 可直接使用               ║
╚══════════════════════════════════════════════════════════════════════════════════╝
""")

## 参考资料

| 资源 | URL |
|------|-----|
| OpenClaw GitHub | https://github.com/openclaw/openclaw |
| OpenClaw 官方文档 | https://openclaw.steephen.cc |
| OpenClaw 社区技能 | https://agentskills.io |
| OpenAI Function Calling 文档 | https://platform.openai.com/docs/guides/function-calling |
| Hermes Agent 官方文档 | https://hermes-agent.nousresearch.com/docs/ |
| Hermes Agent GitHub | https://github.com/NousResearch/hermes-agent |
| LangChain 工具文档 | https://python.langchain.com/docs/concepts/tools/ |
| LangChain GitHub | https://github.com/langchain-ai/langchain |
| Lilian Weng: LLM Powered Autonomous Agents | https://lilianweng.github.io/posts/2023-06-23-agent/ |
| Chip Huyen: The Anatomy of an AI Agent | https://huyenchip.com/2025/01/16/agents.html |

## 6.1 认识 OpenClaw —— SKILL.md 技能生态

OpenClaw 是另一个值得关注的 AI Agent 框架，它奠定了 **SKILL.md 技能格式**的事实标准，与 Hermes Agent 共享同一个技能生态（[agentskills.io](https://agentskills.io)）。

**核心特点**：
- 5,400+ 社区技能，涵盖编程、DevOps、数据分析等多个领域
- 技能以 Markdown 文件（`SKILL.md`）形式存在，易于编写和分享
- 通过 Shell 命令白名单 + API 调用实现工具能力
- 文件式记忆（`MEMORY.md` / `USER.md`）

**与 Hermes Agent 的关系**：Hermes Agent 提供了 `hermes claw migrate` 命令，可以一键导入 OpenClaw 的所有技能、记忆和配置——两者同属一个生态，Hermes 是"进化版"。

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════╗
║            手写 Schema vs LangChain @tool vs OpenClaw SKILL.md 对比              ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  手写 Schema（第3节）：                                                          ║
║  ✅ 完全控制格式和描述                                                          ║
║  ✅ 不依赖任何框架                                                             ║
║  ❌ 代码冗长，容易出错                                                         ║
║  ❌ 需要手动维护 Schema 与 Handler 的同步                                       ║
║                                                                                  ║
║  LangChain @tool（第6节）：                                                     ║
║  ✅ 从函数签名自动生成 Schema                                                   ║
║  ✅ 代码简洁，类型安全                                                         ║
║  ❌ 依赖框架                                                                   ║
║  ❌ 描述格式受限于 docstring                                                   ║
║                                                                                  ║
║  OpenClaw SKILL.md（第6.1节）：                                                  ║
║  ✅ 纯 Markdown，人人都能编写                                                   ║
║  ✅ 包含工作流程、触发条件、步骤清单，更接近"最佳实践"文档                    ║
║  ❌ 无自动 Schema 生成（依赖人工编写）                                         ║
║  ❌ 不直接映射为可执行代码（需配合 Shell/API Handler）                          ║
║                                                                                  ║
║  选择建议：                                                                     ║
║  - 学习阶段：手写 Schema → 理解底层原理                                        ║
║  - 开发阶段：使用框架 @tool → 提高开发效率                                      ║
║  - 知识复用：OpenClaw SKILL.md → 社区技能生态，5,400+ 可直接使用               ║
╚══════════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ==========================================
# OpenClaw 风格：SKILL.md 技能定义示例
# ==========================================

# OpenClaw 的技能以 YAML frontmatter + Markdown 内容组成
# 以下是模拟的 SKILL.md 格式（实际文件存储在 Skills/ 目录下）

skill_example = """---
name: github-issue-summary
description: 当用户要求监控或总结 GitHub 仓库的 Issue 时使用。
version: 1.0.0
---

# GitHub Issue Summary

## Overview
抓取 GitHub 仓库的 open issues，按标签分类并生成摘要报告。

## When to Use
- 用户说"帮我监控 XX 仓库的 issue"
- 用户说"总结最近新增的 bug"
- 不要用于：一次性搜索（用 web_search 即可）

## Steps
1. **Fetch** — `gh issue list --repo <owner/repo> --state open --limit 50`
2. **Filter** — 按标签（bug/feature/help-wanted）分组
3. **Summarize** — 对每个分组生成一句话摘要
4. **Output** — 返回结构化报告

## Required Tools
- gh CLI（需已认证：`gh auth login`）
- web_search（API fallback）
"""

print("✅ OpenClaw SKILL.md 示例")
print("=" * 60)
print(skill_example)

# ==========================================
# OpenClaw 风格：文件式记忆示例
# ==========================================

memory_md_example = """# MEMORY.md — Agent 记忆
- 用户偏好中文回复，技术解释简洁
- 项目使用 pytest 框架，CI 在 GitHub Actions
- 上次调试：API 路由问题在 v2 分支已修复
"""

user_md_example = """# USER.md — 用户画像
- 姓名：张三
- 职业：软件工程师
- 偏好：不喜欢长篇大论，希望直接给出可执行的操作步骤
"""

print("\n📝 MEMORY.md 示例（Agent 跨会话记忆）")
print("-" * 40)
print(memory_md_example)

print("\n👤 USER.md 示例（用户偏好）")
print("-" * 40)
print(user_md_example)

# 模拟：会话开始时将记忆注入系统提示
system_with_memory = f"""你是一个智能助手，可以搜索信息、计算数学表达式、查询时间。
当前用户偏好：简洁回答，直接给出结论。
已知背景：{memory_md_example}
"""
print("\n💡 会话注入示例（系统提示 + 记忆）：")
print(system_with_memory)